# Stage 07 — Stock Movement Analysis

**Input:** `In_and_Out.xlsx` (243,000 rows of SAP material documents)
**Output:** `stock_movements.parquet`, `monthly_demand.parquet`, `ingestion_log.parquet`

---

## Why Stock Movements, Not Orders?

`orders.xlsx` records what dealers *ordered*.
`In_and_Out.xlsx` records what the warehouse *actually issued* — the true demand signal.

The difference matters because:
- A back-ordered part with `confirmed_qty = 0` appears in orders but generates no issue movement
- Internal transfers appear as orders but are not external demand
- Using issues avoids double-counting demand that was partially fulfilled across multiple order cycles

## SAP Movement Type Semantic Classification

| MT Code | Semantic class | Included in demand? |
|---|---|---|
| 601 | Issue (primary demand) | ✅ Yes |
| 101 | Receipt (inbound from supplier) | ✅ Stock reconciliation |
| 651 / 653 | Return from dealer | ✅ Net demand correction |
| 301 / 311 / 641 / 413 | Internal transfer | ❌ No — would double-count |
| 551 | Scrap write-off | ✅ Logged separately |
| 701 / 702 | Inventory adjustment | ✅ Logged separately |

## Append-Only Deduplication

SHA-256 hash of (Material + Posting Date + MT + Qty + Value + Customer).
New month extracts are merged without overwriting historical data.
Duplicate rows (same natural key) are silently dropped — logged in `ingestion_log.parquet`.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE", "#EF4444", "#2CC56F", "#F59E0B", "#A855F7", "#64748B"]

def load(name: str) -> pd.DataFrame:
    for base in [INTERIM, PROCESSED]:
        p = base / name
        if p.exists():
            return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found in interim or processed")


In [ ]:
mv = load("stock_movements.parquet")
md = load("monthly_demand.parquet")
ig = load("ingestion_log.parquet")

print(f"Stock movements : {len(mv):,} rows  |  {mv.shape[1]} columns")
print(f"Monthly demand  : {len(md):,} SKU-month rows")
print(f"Ingestion log   : {len(ig):,} ingestion events")
print()
print("Movement columns:", mv.columns.tolist()[:15])


In [ ]:
mc_col = next((c for c in ["movement_class","movement_type_label","class"] if c in mv.columns), None)
mt_col = next((c for c in ["movement_type","movement_type_code","MT"] if c in mv.columns), None)

if mc_col:
    mc = mv[mc_col].value_counts()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    mc.plot(kind="bar", ax=axes[0], color=PALETTE[:len(mc)], edgecolor="white")
    axes[0].set_title("SAP Movement Documents by Semantic Class")
    axes[0].set_ylabel("Row count"); plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha="right")
    for bar, val in zip(axes[0].patches, mc.values):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200, f"{val:,}",
                     ha="center", va="bottom", fontsize=8)

    mc.plot(kind="pie", ax=axes[1], colors=PALETTE[:len(mc)],
            autopct="%1.1f%%", startangle=90, legend=False)
    axes[1].set_title("Movement Class Share"); axes[1].set_ylabel("")
    plt.tight_layout(); plt.show()

    print("\nMovement class breakdown:")
    print(mc.to_string())
elif mt_col:
    print("Movement type codes (top 15):")
    print(mv[mt_col].value_counts().head(15).to_string())


In [ ]:
# Monthly aggregate demand
month_col = next((c for c in ["month","posting_period","period"] if c in md.columns), None)
qty_col   = next((c for c in ["issue_qty","qty","demand_qty"] if c in md.columns), None)

if month_col and qty_col:
    agg = md.groupby(month_col)[qty_col].agg(["sum","count"]).rename(
        columns={"sum":"total_qty","count":"sku_count"})

    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
    agg["total_qty"].plot(ax=axes[0], color=PALETTE[0], linewidth=2)
    axes[0].set_title("Total Monthly Issue Quantity — All SKUs")
    axes[0].set_ylabel("Units issued")
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

    agg["sku_count"].plot(ax=axes[1], color=PALETTE[2], linewidth=2)
    axes[1].set_title("Number of Active SKUs per Month (at least 1 issue)")
    axes[1].set_ylabel("SKUs with demand"); axes[1].set_xlabel("Month")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout(); plt.show()
else:
    print("monthly_demand columns:", md.columns.tolist())


In [ ]:
# Ingestion log — data quality summary
if len(ig) > 0:
    print("Ingestion log columns:", ig.columns.tolist())
    dup_col = next((c for c in ["duplicates_skipped","duplicate_count","skipped"] if c in ig.columns), None)
    new_col = next((c for c in ["new_rows","rows_inserted","new_records"] if c in ig.columns), None)
    if dup_col or new_col:
        show_cols = [c for c in [new_col, dup_col, "validation_failures","ingested_at"] if c and c in ig.columns]
        print("\nIngestion summary:")
        print(ig[show_cols].tail(10).to_string())
else:
    print("Ingestion log is empty.")


**Output consumed by:** Stage 8 (demand statistics), Stage 11 (stock tracker audit columns: total_receipts, total_issues, last_movement_date), and Stage 12 (ML safety stock features).